# Squirrel YOLO Training for Colab A100 and Raspberry Pi 5 AI HAT+ 26 TOPS

This notebook trains a one-class squirrel detector from `squirrel_dataset_combined_cleaned_removed_blank_055_160.zip`.

Target workflow:

1. Train and validate on Google Colab with an A100 GPU.
2. Export the best model to ONNX with a fixed input size.
3. Create a calibration image pack from your garden/background domain.
4. Use Hailo Dataflow Compiler on x86_64 Linux to compile ONNX to HEF for Raspberry Pi 5 AI HAT+ 26 TOPS.

Important hardware note: Raspberry Pi AI HAT+ 26 TOPS uses Hailo-8, so the Hailo compiler target should be `hailo8`. The older Raspberry Pi AI Kit / 13 TOPS path uses Hailo-8L and `hailo8l`.

Official references used while preparing this notebook:

- Raspberry Pi AI HAT+ docs: https://www.raspberrypi.com/documentation/accessories/ai-hat-plus.html
- Ultralytics Hailo export notes: https://docs.ultralytics.com/integrations/hailo
- Ultralytics export mode: https://docs.ultralytics.com/modes/export
- Hailo Model Zoo: https://github.com/hailo-ai/hailo_model_zoo

## Colab Runtime

Use `Runtime > Change runtime type > GPU`, then select an A100 if your Colab tier offers it.

The defaults below prioritize garden accuracy while still staying realistic for a Hailo-8 26 TOPS edge target:

- `MODEL_WEIGHTS = "yolo11m.pt"` for accuracy.
- `IMG_SIZE = 960` so small squirrels in grass/trees keep more pixels.
- If Hailo compilation or Pi latency is not acceptable, rerun with `MODEL_WEIGHTS = "yolo11s.pt"` and optionally `IMG_SIZE = 640`.

In [ ]:
!nvidia-smi
!python --version
!pip -q install -U ultralytics pyyaml opencv-python-headless matplotlib pandas seaborn onnx onnxruntime

Thu Jul  9 23:02:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from pathlib import Path
import os
import random
import shutil
import zipfile
import json
import yaml
import math

import cv2
import numpy as np
import matplotlib.pyplot as plt

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Upload this zip to Colab, or put it in Google Drive and copy it to this path.
DATASET_ZIP = Path("/content/squirrel_dataset_combined_cleaned_removed_blank_055_160.zip")
DATA_ROOT = Path("/content/datasets/squirrel_cleaned")

# High accuracy profile. For the fastest Hailo compile/runtime fallback, use "yolo11s.pt".
MODEL_WEIGHTS = os.environ.get("MODEL_WEIGHTS", "yolo11m.pt")
IMG_SIZE = int(os.environ.get("IMG_SIZE", "960"))  # fixed export size; must be divisible by 32
EPOCHS = int(os.environ.get("EPOCHS", "200"))
BATCH = int(os.environ.get("BATCH", "24"))  # A100 should handle this for YOLO11m at 960; lower to 12 if OOM.
PATIENCE = int(os.environ.get("PATIENCE", "45"))
WORKERS = int(os.environ.get("WORKERS", "8"))

PROJECT = Path("/content/runs/squirrel_yolo")
RUN_NAME = f"{Path(MODEL_WEIGHTS).stem}_img{IMG_SIZE}_garden"

assert IMG_SIZE % 32 == 0, "IMG_SIZE should be divisible by 32 for YOLO export/deployment."
print({
    "dataset_zip": str(DATASET_ZIP),
    "model": MODEL_WEIGHTS,
    "imgsz": IMG_SIZE,
    "epochs": EPOCHS,
    "batch": BATCH,
    "run_name": RUN_NAME,
})

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{'dataset_zip': '/content/squirrel_dataset_combined_cleaned_removed_blank_055_160.zip', 'model': 'yolo11m.pt', 'imgsz': 960, 'epochs': 200, 'batch': 24, 'run_name': 'yolo11m_img960_garden'}


## Upload or Mount the Dataset

If the zip is not already at `/content/squirrel_dataset_combined_cleaned_removed_blank_055_160.zip`, run the upload cell.

For a long Colab session, Google Drive is usually better than browser upload. If you use Drive, copy the zip to `DATASET_ZIP` before continuing.

In [ ]:
if not DATASET_ZIP.exists():
    from google.colab import files
    print("Upload squirrel_dataset_combined_cleaned_removed_blank_055_160.zip")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise FileNotFoundError("No zip file was uploaded.")
    uploaded_zip = Path(zip_names[0])
    shutil.move(str(uploaded_zip), DATASET_ZIP)

print("Dataset zip:", DATASET_ZIP, "size MB:", round(DATASET_ZIP.stat().st_size / 1_000_000, 2))

Upload squirrel_dataset_combined_cleaned_removed_blank_055_160.zip


In [ ]:
# Optional Google Drive flow.
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copy2("/content/drive/MyDrive/squirrel_dataset_combined_cleaned_removed_blank_055_160.zip", DATASET_ZIP)

## Unzip and Normalize `data.yaml`

This cell does not trust relative paths from the zip. It finds the YOLO `images/train`, `images/val`, `labels/train`, and `labels/val` folders, then writes a Colab-safe YAML.

In [ ]:
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(DATASET_ZIP, "r") as z:
    z.extractall(DATA_ROOT)

def find_dataset_root(base: Path) -> Path:
    matches = sorted(base.rglob("images/train"))
    if not matches:
        raise FileNotFoundError("Could not find images/train inside the dataset zip.")
    return matches[0].parent.parent

YOLO_ROOT = find_dataset_root(DATA_ROOT)
print("YOLO root:", YOLO_ROOT)

classes_txt = YOLO_ROOT / "classes.txt"
if classes_txt.exists():
    names = [line.strip() for line in classes_txt.read_text(encoding="utf-8", errors="replace").splitlines() if line.strip()]
else:
    names = ["squirrel"]
if not names:
    names = ["squirrel"]

data = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {i: name for i, name in enumerate(names)},
}

DATA_YAML = YOLO_ROOT / "data_colab.yaml"
DATA_YAML.write_text(yaml.safe_dump(data, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text())

## Dataset Audit

YOLO can use blank label files as real background images. This dataset intentionally keeps the first 54 blank-label backgrounds and removes the blank duplicate set.

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def expected_label_for_image(img_path: Path) -> Path:
    parts = list(img_path.parts)
    idx = parts.index("images")
    parts[idx] = "labels"
    return Path(*parts).with_suffix(".txt")

def audit_split(split: str):
    img_dir = YOLO_ROOT / "images" / split
    images = sorted(p for p in img_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
    missing = []
    blank = []
    labeled = []
    box_count = 0
    for img in images:
        lbl = expected_label_for_image(img)
        if not lbl.exists():
            missing.append((img, lbl))
            continue
        lines = [line for line in lbl.read_text(encoding="utf-8", errors="replace").splitlines() if line.strip()]
        if lines:
            labeled.append(img)
            box_count += len(lines)
        else:
            blank.append(img)
    return {
        "split": split,
        "images": len(images),
        "labeled_images": len(labeled),
        "blank_background_images": len(blank),
        "boxes": box_count,
        "missing_labels": len(missing),
    }

summary = [audit_split("train"), audit_split("val")]
print(json.dumps(summary, indent=2))
assert all(item["missing_labels"] == 0 for item in summary), "Some images are missing labels."

In [ ]:
def read_yolo_boxes(label_path: Path):
    rows = []
    if not label_path.exists():
        return rows
    for line in label_path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.strip():
            continue
        cls, x, y, w, h = line.split()[:5]
        rows.append((int(float(cls)), float(x), float(y), float(w), float(h)))
    return rows

def draw_boxes_rgb(img_path: Path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl = expected_label_for_image(img_path)
    for cls, xc, yc, bw, bh in read_yolo_boxes(lbl):
        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, names[cls], (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return img

labeled_imgs = []
background_imgs = []
for split in ["train", "val"]:
    for img in sorted((YOLO_ROOT / "images" / split).rglob("*")):
        if img.suffix.lower() not in IMAGE_EXTS:
            continue
        boxes = read_yolo_boxes(expected_label_for_image(img))
        (labeled_imgs if boxes else background_imgs).append(img)

sample = random.sample(labeled_imgs, min(6, len(labeled_imgs))) + random.sample(background_imgs, min(3, len(background_imgs)))
plt.figure(figsize=(16, 10))
for i, img_path in enumerate(sample, 1):
    plt.subplot(3, 3, i)
    plt.imshow(draw_boxes_rgb(img_path))
    plt.title(img_path.name[:38], fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Train

The settings below are tuned for a small outdoor animal detector:

- Pretrained YOLO weights for general visual features.
- Larger fixed image size for small/camouflaged squirrels.
- Moderate color and geometric augmentation for changing garden lighting.
- Mosaic closes near the end so final epochs see natural images.
- Early stopping prevents overtraining if validation stops improving.

In [ ]:
model = YOLO(MODEL_WEIGHTS)

results = model.train(
    data=str(DATA_YAML),
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH,
    device=0,
    workers=WORKERS,
    project=str(PROJECT),
    name=RUN_NAME,
    pretrained=True,
    seed=SEED,
    patience=PATIENCE,
    optimizer="AdamW",
    lr0=0.002,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    cos_lr=True,
    close_mosaic=20,
    mosaic=0.60,
    mixup=0.05,
    copy_paste=0.0,
    hsv_h=0.015,
    hsv_s=0.50,
    hsv_v=0.35,
    degrees=5.0,
    translate=0.08,
    scale=0.45,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,
    cache="ram",
    amp=True,
    plots=True,
    exist_ok=True,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
print("Run dir:", RUN_DIR)
print("Best weights:", BEST_PT)

## Optional High-Resolution Finish

Run this only if the validation set is still missing small squirrels and you can afford extra training time. It fine-tunes the best checkpoint at the same or higher input size with mosaic disabled.

In [ ]:
RUN_FINE_TUNE = False
FINE_TUNE_IMG_SIZE = IMG_SIZE  # try 1280 only if you are okay with slower edge inference/compile.

if RUN_FINE_TUNE:
    ft_model = YOLO(str(BEST_PT))
    ft_results = ft_model.train(
        data=str(DATA_YAML),
        imgsz=FINE_TUNE_IMG_SIZE,
        epochs=50,
        batch=max(4, BATCH // 2),
        device=0,
        workers=WORKERS,
        project=str(PROJECT),
        name=f"{RUN_NAME}_finetune",
        seed=SEED,
        patience=20,
        optimizer="AdamW",
        lr0=0.0005,
        lrf=0.01,
        weight_decay=0.0005,
        warmup_epochs=1.0,
        cos_lr=True,
        mosaic=0.0,
        mixup=0.0,
        close_mosaic=0,
        hsv_h=0.010,
        hsv_s=0.35,
        hsv_v=0.25,
        translate=0.04,
        scale=0.25,
        fliplr=0.5,
        cache="ram",
        amp=True,
        plots=True,
        exist_ok=True,
    )
    RUN_DIR = Path(ft_model.trainer.save_dir)
    BEST_PT = RUN_DIR / "weights" / "best.pt"
    print("Fine-tuned best:", BEST_PT)

## Validate and Inspect Predictions

For garden monitoring, the right confidence threshold is usually lower than a demo threshold. Start around `0.20`, then raise it if you get too many false positives.

In [ ]:
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(
    data=str(DATA_YAML),
    imgsz=IMG_SIZE,
    device=0,
    conf=0.001,
    iou=0.70,
    plots=True,
    save_json=True,
)
print(metrics)

In [ ]:
PRED_CONF = 0.20
PRED_IOU = 0.45

val_images = sorted((YOLO_ROOT / "images" / "val").glob("*"))
pred_source = random.sample(val_images, min(24, len(val_images)))

pred = best_model.predict(
    source=[str(p) for p in pred_source],
    imgsz=IMG_SIZE,
    conf=PRED_CONF,
    iou=PRED_IOU,
    max_det=10,
    device=0,
    save=True,
    project=str(PROJECT),
    name=f"{RUN_NAME}_val_predictions",
    exist_ok=True,
)
print("Saved predictions under:", PROJECT)

## Test on Your Own Garden Photos or Clips

Upload a few real garden images/videos from the exact camera angle you plan to use. This is the fastest way to catch misses from distance, motion blur, shade, fence lines, wet grass, or nighttime lighting.

In [ ]:
RUN_GARDEN_UPLOAD_TEST = False

if RUN_GARDEN_UPLOAD_TEST:
    from google.colab import files
    upload_dir = Path("/content/garden_test_uploads")
    upload_dir.mkdir(exist_ok=True)
    uploaded = files.upload()
    for name in uploaded:
        shutil.move(name, upload_dir / name)
    best_model.predict(
        source=str(upload_dir),
        imgsz=IMG_SIZE,
        conf=PRED_CONF,
        iou=PRED_IOU,
        max_det=20,
        device=0,
        save=True,
        project=str(PROJECT),
        name=f"{RUN_NAME}_garden_upload_predictions",
        exist_ok=True,
    )

## Export for A100 Validation and Hailo Handoff

For Hailo, export a static ONNX with a fixed `imgsz`. Do not use dynamic shapes for the NPU build.

The Hailo HEF compiler stage is separate:

- It requires Hailo Dataflow Compiler.
- Hailo's docs say DFC runs on Linux x86_64 and not on ARM Raspberry Pi.
- If you have the DFC wheel available inside Colab, you can compile here; otherwise compile on an x86_64 Ubuntu machine, then copy the `.hef` and `metadata.yaml` to the Pi.

In [ ]:
EXPORT_DIR = RUN_DIR / "export"
EXPORT_DIR.mkdir(exist_ok=True)

best_model = YOLO(str(BEST_PT))

ONNX_PATH = Path(best_model.export(
    format="onnx",
    imgsz=IMG_SIZE,
    opset=11,       # broad Hailo DFC compatibility
    simplify=True,
    dynamic=False,
    nms=False,
))

copied_onnx = EXPORT_DIR / ONNX_PATH.name
if ONNX_PATH.resolve() != copied_onnx.resolve():
    shutil.copy2(ONNX_PATH, copied_onnx)

shutil.copy2(BEST_PT, EXPORT_DIR / BEST_PT.name)
shutil.copy2(DATA_YAML, EXPORT_DIR / "data_colab.yaml")

print("ONNX:", copied_onnx)
print("PT:", EXPORT_DIR / BEST_PT.name)

In [ ]:
# Optional: TensorRT engine for Colab/A100 inference benchmarking. This is not for Raspberry Pi/Hailo.
RUN_TENSORRT_EXPORT = False

if RUN_TENSORRT_EXPORT:
    engine_path = best_model.export(
        format="engine",
        imgsz=IMG_SIZE,
        half=True,
        dynamic=False,
        batch=1,
        device=0,
    )
    print("TensorRT engine:", engine_path)

## Build Hailo Calibration Image Pack

Use garden-domain images for INT8 calibration. The cell below samples training images, preferring images that actually contain squirrels, then adds some blank background frames.

In [ ]:
CALIB_DIR = EXPORT_DIR / "hailo_calibration_images"
if CALIB_DIR.exists():
    shutil.rmtree(CALIB_DIR)
CALIB_DIR.mkdir(parents=True, exist_ok=True)

labeled = []
blank = []
for img in sorted((YOLO_ROOT / "images" / "train").rglob("*")):
    if img.suffix.lower() not in IMAGE_EXTS:
        continue
    if read_yolo_boxes(expected_label_for_image(img)):
        labeled.append(img)
    else:
        blank.append(img)

random.shuffle(labeled)
random.shuffle(blank)
calib_images = (labeled[:900] + blank[:124])[:1024]
if len(calib_images) < 64:
    raise RuntimeError("Need at least 64 calibration images.")

for i, src in enumerate(calib_images):
    dst = CALIB_DIR / f"calib_{i:04d}{src.suffix.lower()}"
    shutil.copy2(src, dst)

print("Calibration images:", len(list(CALIB_DIR.iterdir())))
print("Calibration dir:", CALIB_DIR)

## Hailo HEF Compile Notes for Raspberry Pi 5 AI HAT+ 26 TOPS

Use `HW_ARCH = "hailo8"` for the 26 TOPS AI HAT+.

The cell below writes a starter compile script into the export folder. It is intentionally guarded because custom YOLO models need a matching NMS JSON and sometimes model-script layer names adjusted from the DFC parse log.

Expected Hailo flow:

1. Activate Hailo Dataflow Compiler on x86_64 Linux.
2. Copy the export folder there.
3. Provide an NMS JSON matching this exact model, class count, and `IMG_SIZE`.
4. Run the generated script.
5. Copy the resulting `.hef` and `metadata.yaml` to the Raspberry Pi.

In [ ]:
HAILO_COMPILE_SCRIPT = EXPORT_DIR / "compile_hailo8_squirrel.py"
script = f'''\
from pathlib import Path
import ast
import random
import numpy as np
import onnx
import yaml
from PIL import Image
from hailo_sdk_client import ClientRunner

MODEL_NAME = "squirrel_yolo11_custom"
HW_ARCH = "hailo8"  # Raspberry Pi AI HAT+ 26 TOPS. Use "hailo8l" only for 13 TOPS / AI Kit.
IMGSZ = {IMG_SIZE}
ONNX_PATH = Path(r"{copied_onnx.name}")
CALIB_DIR = Path("hailo_calibration_images")
NMS_CONFIG = Path("squirrel_yolo11_nms_config.json")
OUT_DIR = Path("hailo8_squirrel_model")
OUT_DIR.mkdir(exist_ok=True)

if not NMS_CONFIG.exists():
    raise FileNotFoundError(
        "Missing squirrel_yolo11_nms_config.json. Generate or adapt it for this exact YOLO11 model, "
        "1 class, strides, end nodes, and fixed image size before compiling."
    )

END_NODES = [
    "/model.23/cv2.0/cv2.0.2/Conv",
    "/model.23/cv3.0/cv3.0.2/Conv",
    "/model.23/cv2.1/cv2.1.2/Conv",
    "/model.23/cv3.1/cv3.1.2/Conv",
    "/model.23/cv2.2/cv2.2.2/Conv",
    "/model.23/cv3.2/cv3.2.2/Conv",
]

meta = {{p.key: p.value for p in onnx.load(ONNX_PATH, load_external_data=False).metadata_props}}
for k in ("stride", "batch", "channels"):
    if k in meta:
        meta[k] = int(meta[k])
for k in ("imgsz", "names", "args", "end2end"):
    if k in meta:
        meta[k] = ast.literal_eval(meta[k])
with open(OUT_DIR / "metadata.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(meta, f, sort_keys=False)

runner = ClientRunner(hw_arch=HW_ARCH)
runner.translate_onnx_model(str(ONNX_PATH), end_node_names=END_NODES)

# These conv names are assigned during DFC parsing and may need adjustment for a custom model.
# If DFC errors here, read the parse log and update conv layer names.
model_script = (
    "normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])\\n"
    "change_output_activation(conv54, sigmoid)\\n"
    "change_output_activation(conv65, sigmoid)\\n"
    "change_output_activation(conv80, sigmoid)\\n"
    f'nms_postprocess("{{NMS_CONFIG}}", meta_arch=yolov8, engine=cpu)\\n'
    "allocator_param(width_splitter_defuse=disabled)"
)
runner.load_model_script(model_script)

image_files = sorted([p for p in CALIB_DIR.iterdir() if p.suffix.lower() in {{".jpg", ".jpeg", ".png", ".bmp", ".webp"}}])
if len(image_files) < 64:
    raise RuntimeError("Calibration set is too small. Use at least 64 representative garden images.")

calib_count = min(1024, len(image_files))
calibset = np.zeros((calib_count, IMGSZ, IMGSZ, 3), dtype=np.float32)
for i, img_path in enumerate(random.sample(image_files, calib_count)):
    img = Image.open(img_path).convert("RGB").resize((IMGSZ, IMGSZ))
    calibset[i] = np.array(img, dtype=np.float32)

runner.optimize(calibset)
runner.save_har(str(OUT_DIR / f"{{MODEL_NAME}}.o.har"))
hef = runner.compile()
hef_path = OUT_DIR / f"{{MODEL_NAME}}.hef"
with open(hef_path, "wb") as f:
    f.write(hef)
print("Compiled HEF:", hef_path)
print("Keep metadata next to HEF:", OUT_DIR / "metadata.yaml")
'''
HAILO_COMPILE_SCRIPT.write_text(script, encoding="utf-8")
print(HAILO_COMPILE_SCRIPT)
print(HAILO_COMPILE_SCRIPT.read_text()[:1500])

## Package Training Artifacts

The final zip contains:

- Best PyTorch weights.
- ONNX export for Hailo handoff.
- Colab-safe `data_colab.yaml`.
- Calibration images for Hailo INT8 quantization.
- Training metrics/plots from the run directory.
- Starter `compile_hailo8_squirrel.py`.

In [ ]:
ARTIFACT_ZIP = Path("/content/squirrel_yolo_colab_a100_hailo8_artifacts.zip")
if ARTIFACT_ZIP.exists():
    ARTIFACT_ZIP.unlink()

shutil.make_archive(str(ARTIFACT_ZIP.with_suffix("")), "zip", RUN_DIR)
print("Artifact zip:", ARTIFACT_ZIP, "size MB:", round(ARTIFACT_ZIP.stat().st_size / 1_000_000, 2))

from google.colab import files
files.download(str(ARTIFACT_ZIP))

## Raspberry Pi Runtime Handoff

After Hailo compilation, the Pi needs the `.hef` and `metadata.yaml` together.

On Raspberry Pi OS with AI HAT+ 26 TOPS, install the Hailo packages and check the device:

```bash
sudo apt update
sudo apt install -y dkms hailo-all
sudo reboot
hailortcli fw-control identify
```

Then use HailoRT or Picamera2/Hailo examples to run the HEF on your camera stream. Start with a lower display threshold such as `0.20` for squirrels far away in grass, then raise it if garden background false positives are too common.